## Description

This notebook analyzes poverty convergence and its link to GDP. It examines beta/sigma convergence, with an ARIMA forecast for poverty rate dispersion. Key findings and policy implications are discussed using World Bank data and statistical models.

In [ ]:
# =============================================================================
# POVERTY CONVERGENCE ANALYSIS — Google Colab
# World Bank WDI  |  SI.POV.DDAY  |  $3.00/day (2021 PPP)
#
# Generates:
#   fig5_beta_convergence.png
#   fig6_sigma_convergence.png
#
# INSTRUCTIONS
# 1. Upload your XLS file to Colab:
#       from google.colab import files
#       files.upload()          # select API_SI_POV_DDAY_DS2_en_excel_v2_261020.xls
# 2. Run all cells in order.
# 3. Figures are saved to your Colab session and downloaded automatically.
# =============================================================================

# ── CELL 1 · Install / import ─────────────────────────────────────────────────
# !pip install xlrd seaborn --quiet

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from scipy import stats

import matplotlib
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
import matplotlib.lines as mlines
from matplotlib.ticker import MultipleLocator
import seaborn as sns

print(f"matplotlib {matplotlib.__version__}  |  seaborn {sns.__version__}")


# ── CELL 2 · Colour palette & style ──────────────────────────────────────────
WB_BLUE   = '#1F5DAD'
WB_RED    = '#BE2627'
WB_ORANGE = '#E07B39'
WB_AMBER  = '#C4A227'
WB_PURPLE = '#5C4B8A'
WB_GREEN  = '#3DAD6E'
WB_GREY   = '#6D6E71'
WB_LGREY  = '#D9D9D9'
WB_PANEL  = '#F5F5F5'
WB_BG     = '#FFFFFF'
TEXT_COL  = '#231F20'
MUTED     = '#888888'
GRID_COL  = '#EBEBEB'

REGION_COLORS = {
    'East Asia & Pacific':        WB_BLUE,
    'Europe & Central Asia':      WB_GREEN,
    'Latin America & Caribbean':  WB_ORANGE,
    'Middle East & North Africa': WB_AMBER,
    'South Asia':                 WB_PURPLE,
    'Sub-Saharan Africa':         WB_RED,
}
REG_SHORT = {
    'East Asia & Pacific':        'EAP',
    'Europe & Central Asia':      'ECA',
    'Latin America & Caribbean':  'LAC',
    'Middle East & North Africa': 'MENA',
    'South Asia':                 'SAS',
    'Sub-Saharan Africa':         'SSA',
}
REGION_CODES = {
    'East Asia & Pacific':        'EAS',
    'Europe & Central Asia':      'ECS',
    'Latin America & Caribbean':  'LCN',
    'Middle East & North Africa': 'MEA',
    'South Asia':                 'SAS',
    'Sub-Saharan Africa':         'SSF',
}
IG_COLORS = {
    'Low income':          WB_RED,
    'Lower middle income': WB_ORANGE,
    'Upper middle income': WB_BLUE,
    'High income':         WB_GREEN,
}

# Global Matplotlib / Seaborn style
plt.rcParams.update({
    'font.family':        'DejaVu Sans',
    'axes.facecolor':     WB_BG,
    'figure.facecolor':   WB_BG,
    'axes.edgecolor':     WB_LGREY,
    'axes.linewidth':     0.8,
    'axes.grid':          True,
    'grid.color':         GRID_COL,
    'grid.linewidth':     0.55,
    'grid.linestyle':     '-',
    'axes.axisbelow':     True,
    'xtick.labelsize':    9,
    'ytick.labelsize':    9,
    'xtick.color':        MUTED,
    'ytick.color':        MUTED,
    'xtick.major.size':   3,
    'ytick.major.size':   3,
    'axes.spines.top':    False,
    'axes.spines.right':  False,
    'legend.frameon':     False,
    'legend.fontsize':    9,
    'savefig.dpi':        180,
    'savefig.bbox':       'tight',
    'savefig.facecolor':  WB_BG,
})
sns.set_style('whitegrid', {
    'axes.facecolor':  WB_BG,
    'grid.color':      GRID_COL,
    'axes.edgecolor':  WB_LGREY,
})


# ── CELL 3 · Helper functions ─────────────────────────────────────────────────
def hp_filter(y, lamb=100):
    """Hodrick-Prescott filter. Returns (trend, cycle)."""
    n  = len(y)
    I  = np.eye(n)
    D2 = np.zeros((n - 2, n))
    for i in range(n - 2):
        D2[i, i] = 1; D2[i, i + 1] = -2; D2[i, i + 2] = 1
    trend = np.linalg.solve(I + lamb * D2.T @ D2, y)
    return trend, y - trend


def ols(x, y):
    """OLS. Returns (intercept, slope, se_int, se_slope, t, p, r2, y_hat, resid)."""
    X  = np.column_stack([np.ones(len(x)), x])
    b, _, _, _ = np.linalg.lstsq(X, y, rcond=None)
    yh = X @ b; e = y - yh; n, k = len(y), 2
    s2 = np.sum(e ** 2) / (n - k)
    se = np.sqrt(np.diag(s2 * np.linalg.inv(X.T @ X)))
    t  = b[1] / se[1]
    p  = 2 * (1 - stats.t.cdf(abs(t), n - k))
    r2 = 1 - np.sum(e ** 2) / np.sum((y - y.mean()) ** 2)
    return b[0], b[1], se[0], se[1], t, p, r2, yh, e


def sig_stars(p):
    return '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'ns'


def style_ax(ax, ylabel='', xlabel=''):
    ax.spines['left'].set_color(WB_LGREY)
    ax.spines['bottom'].set_color(WB_LGREY)
    if ylabel: ax.set_ylabel(ylabel, fontsize=9, color=MUTED, labelpad=5)
    if xlabel: ax.set_xlabel(xlabel, fontsize=9, color=MUTED, labelpad=5)


def panel_label(ax, letter, title, fontsize=10):
    ax.set_title(f'{letter}.  {title}', fontsize=fontsize, fontweight='bold',
                 color=TEXT_COL, pad=10, loc='left', fontfamily='DejaVu Serif')


def source_note(fig, text, y=0.01):
    fig.text(0.01, y, text, fontsize=7.2, color=MUTED,
             style='italic', ha='left', va='bottom', wrap=True)


# ── CELL 4 · Load & prepare data ─────────────────────────────────────────────
DATA_PATH = '/content/API_SI.POV.DDAY_DS2_en_excel_v2_261020.xls'   # <-- adjust if needed

df_raw = pd.read_excel(DATA_PATH, sheet_name='Data', engine='xlrd', header=3)
df_raw.columns = [str(c).replace('.0', '') for c in df_raw.columns]
meta   = pd.read_excel(DATA_PATH, sheet_name='Metadata - Countries', engine='xlrd')
df_raw = df_raw.merge(
    meta[['Country Code', 'Region', 'IncomeGroup']], on='Country Code', how='left'
)
year_cols = [c for c in df_raw.columns if c.isdigit() and 1990 <= int(c) <= 2023]
countries = df_raw[df_raw['Country Code'].isin(meta['Country Code'].values)].copy()

# ── Beta-convergence dataset ──────────────────────────────────────────────────
rows = []
for _, row in countries.iterrows():
    available = [(int(y), float(row[y])) for y in year_cols if pd.notna(row[y])]
    if len(available) < 2: continue
    available.sort()
    y0, p0 = available[0]; y1, p1 = available[-1]
    T = y1 - y0
    if T < 5 or p0 < 1.0: continue
    rows.append({
        'country':    row['Country Name'],
        'code':       row['Country Code'],
        'region':     row.get('Region', ''),
        'income':     row.get('IncomeGroup', ''),
        'p0': p0, 'p1': p1, 'T': T,
        'y_start': y0, 'y_end': y1,
        'ann_change': (p1 - p0) / T,
        'log_p0':     np.log(p0),
    })
dfc = pd.DataFrame(rows)

# Global OLS
a0, a1, se0, se1, t_b, p_b, r2_g, yh_g, resid_g = ols(
    dfc['log_p0'].values, dfc['ann_change'].values
)
dfc['residual'] = resid_g
dfc['fitted']   = yh_g

# Outlier labels (IQR rule + named countries)
q75, q25 = np.percentile(resid_g, 75), np.percentile(resid_g, 25)
iqr       = q75 - q25
named     = {'China', 'Uzbekistan', 'Mali', 'Congo, Dem. Rep.', 'Mozambique', 'India'}
outlier   = dfc[(np.abs(resid_g) > 1.8 * iqr) | dfc['country'].isin(named)].copy()

# Fit line + 95% CI
x_fit   = np.linspace(dfc['log_p0'].min(), dfc['log_p0'].max(), 300)
y_fit   = a0 + a1 * x_fit
ci_se   = np.sqrt(se0 ** 2 + (x_fit ** 2) * se1 ** 2)
y_upper = y_fit + 1.96 * ci_se
y_lower = y_fit - 1.96 * ci_se

# Regional OLS
reg_res = {}
for reg in dfc['region'].dropna().unique():
    sub = dfc[dfc['region'] == reg]
    if len(sub) < 4: continue
    ra0, ra1, _, rse1, rt, rp, rr2, _, _ = ols(
        sub['log_p0'].values, sub['ann_change'].values
    )
    reg_res[reg] = {'beta': ra1, 'se': rse1, 't': rt, 'p': rp, 'r2': rr2, 'N': len(sub)}

# ── Sigma-convergence dataset ─────────────────────────────────────────────────
c_counts = countries[year_cols].notna().sum(axis=1)
valid_c  = countries[c_counts >= 10]['Country Code'].values
sigma_rows = []
for yc in year_cols:
    sub  = countries[countries['Country Code'].isin(valid_c)]
    vals = sub[yc].dropna().values
    if len(vals) >= 20:
        sigma_rows.append({
            'year':  int(yc),
            'sigma': float(np.std(vals)),
            'cv':    float(np.std(vals) / np.mean(vals)),
            'n':     int(len(vals)),
        })
sigma_df = pd.DataFrame(sigma_rows).sort_values('year').reset_index(drop=True)
sl_s, ic_s, r_s, p_s, _ = stats.linregress(sigma_df['year'], sigma_df['sigma'])

# ── Regional HP data ──────────────────────────────────────────────────────────
reg_hp = {}
for name, code in REGION_CODES.items():
    row = df_raw[df_raw['Country Code'] == code]
    if len(row) == 0: continue
    row  = row.iloc[0]
    pts  = sorted({int(y): float(row[y]) for y in year_cols if pd.notna(row[y])}.items())
    yrs  = [p[0] for p in pts]
    vals = np.array([p[1] for p in pts])
    t, c = hp_filter(vals, 100)
    reg_hp[name] = {'years': yrs, 'obs': vals, 'trend': t, 'cycle': c}

print("Data ready.")
print(f"  Beta dataset  : {len(dfc)} countries")
print(f"  Sigma dataset : {len(sigma_df)} year-observations")
print(f"  Regions (HP)  : {len(reg_hp)}")


# ── CELL 5 · Figure 5 — Beta-Convergence ─────────────────────────────────────
#
#   Layout: 3 rows, each full width
#     Row 1 (tall) : scatter — annualized change vs ln(p0)
#     Row 2        : horizontal bar — regional β coefficients
#     Row 3        : bubble chart — β vs R², size = N
#
fig5 = plt.figure(figsize=(14, 18), facecolor=WB_BG)
gs5  = gridspec.GridSpec(
    3, 1, figure=fig5,
    left=0.09, right=0.94, top=0.93, bottom=0.06,
    hspace=0.9, # Increased hspace
    height_ratios=[1.3, 1.0, 1.0],
)

# Header
fig5.text(0.09, 0.965,
    'Beta-Convergence in Global Poverty Reduction, 1990–2023',
    fontsize=14, fontweight='bold', color=TEXT_COL, fontfamily='DejaVu Serif')
fig5.text(0.09, 0.950,
    'OLS regression of annualized poverty change on log initial poverty level  |  N = 140 countries',
    fontsize=9, color=MUTED)

# Shared region legend (top of figure)
legend_handles = [
    mpatches.Patch(color=REGION_COLORS[r],
                   label=f"{REG_SHORT[r]}  {r}")
    for r in REGION_COLORS
]
fig5.legend(
    handles=legend_handles,
    loc='upper right', bbox_to_anchor=(0.94, 0.956),
    ncol=2, fontsize=8.5,
    frameon=True, framealpha=0.95, edgecolor=WB_LGREY,
    handlelength=1.0,
)

# ── Row 1: scatter ────────────────────────────────────────────────────────────
ax1 = fig5.add_subplot(gs5[0])

for reg, col in REGION_COLORS.items():
    sub = dfc[dfc['region'] == reg]
    ax1.scatter(sub['log_p0'], sub['ann_change'],
                color=col, alpha=0.60, s=32, zorder=4,
                edgecolors='none', rasterized=True)

# CI band
ax1.fill_between(x_fit, y_lower, y_upper,
                 color=TEXT_COL, alpha=0.07, zorder=2)

# Fit line
ax1.plot(x_fit, y_fit, color=TEXT_COL, lw=2.0, zorder=5,
         label=f'OLS fit:  β = {a1:.3f}   t = {t_b:.2f}   R² = {r2_g:.3f}   p < 0.001')

# Zero reference
ax1.axhline(0, color=WB_LGREY, lw=1.0, linestyle='--', zorder=1)

# Outlier annotations — manual offsets to prevent overlap
OFFSETS = {
    'China':             ( 0.06,  0.06),
    'Uzbekistan':        ( 0.06, -0.16),
    'Mali':              (-0.06, -0.22),
    'Congo, Dem. Rep.':  ( 0.06,  0.08),
    'Mozambique':        ( 0.06,  0.06),
    'India':             (-1.20, -0.20),
}
for _, row in outlier.iterrows():
    col  = REGION_COLORS.get(row['region'], WB_GREY)
    dx, dy = OFFSETS.get(row['country'], (0.06, 0.06))
    ax1.annotate(
        row['country'],
        xy=(row['log_p0'], row['ann_change']),
        xytext=(row['log_p0'] + dx, row['ann_change'] + dy),
        fontsize=8, color=col,
        arrowprops=dict(arrowstyle='-', color=col,
                        lw=0.6, shrinkA=2, shrinkB=2),
    )

ax1.legend(fontsize=9, loc='lower right', frameon=True,
           framealpha=0.9, edgecolor=WB_LGREY)
panel_label(ax1, 'A',
    'Annualized poverty change vs. initial poverty level  '
    '— negative β confirms absolute convergence')
style_ax(ax1,
    ylabel='Annualized change in poverty (pp / yr)',
    xlabel='Log initial poverty rate  ln(p₀)')
ax1.spines['left'].set_color(WB_LGREY)
ax1.spines['bottom'].set_color(WB_LGREY)

# Inset stats box
stats_txt = (
    'β = −0.409\n'
    't  = −7.02\n'
    'p  < 0.001\n'
    'R² = 0.263\n'
    'N  = 140'
)
ax1.text(
    0.02, 0.05, stats_txt,
    transform=ax1.transAxes,
    fontsize=8.5, color=TEXT_COL, va='bottom', ha='left',
    fontfamily='DejaVu Sans Mono',
    bbox=dict(boxstyle='round,pad=0.5', facecolor=WB_PANEL,
              edgecolor=WB_LGREY, linewidth=0.8),
)

# ── Row 2: regional β bars ────────────────────────────────────────────────────
ax2 = fig5.add_subplot(gs5[1])

reg_order  = sorted(reg_res.keys(), key=lambda r: reg_res[r]['beta'])
reg_betas  = [reg_res[r]['beta'] for r in reg_order]
reg_ses    = [reg_res[r]['se']   for r in reg_order]
reg_ps     = [reg_res[r]['p']    for r in reg_order]
reg_ns     = [reg_res[r]['N']    for r in reg_order]
reg_r2s    = [reg_res[r]['r2']   for r in reg_order]
reg_cols   = [REGION_COLORS.get(r, WB_GREY) for r in reg_order]
ylabels    = [
    f"{REG_SHORT[r]}  "
    + r.replace('Latin America & Caribbean', 'Latin Am. & Caribbean')
     .replace('Middle East & North Africa', 'Middle East & N. Africa')
    for r in reg_order
]
x_pos = np.arange(len(reg_order))

ax2.barh(x_pos, [abs(v) for v in reg_betas],
         color=reg_cols, height=0.52, zorder=3, edgecolor='none')
ax2.errorbar(
    x_pos, [abs(v) for v in reg_betas],
    xerr=[1.96 * s for s in reg_ses],
    fmt='none', color=TEXT_COL, lw=1.4, capsize=6, zorder=5,
)

# Global β reference line
ax2.axvline(abs(a1), color=TEXT_COL, lw=1.4, linestyle='--',
            zorder=6, alpha=0.75)
ax2.text(abs(a1) + 0.012, len(reg_order) - 0.55,
         f'Global β = {abs(a1):.3f}',
         fontsize=8.5, color=TEXT_COL, style='italic', va='top')

# Value + significance labels — placed after the CI whisker
# Calculate max whisker end for each bar to set text x-position safely
for i, (v, p_val, s, n) in enumerate(zip(reg_betas, reg_ps, reg_ses, reg_ns)):
    whisker_end = abs(v) + 1.96 * s
    ax2.text(
        whisker_end + 0.025, i,
        f'{abs(v):.3f} {sig_stars(p_val)}   (N = {n})',
        va='center', fontsize=9, color=TEXT_COL,
    )

ax2.set_yticks(x_pos)
ax2.set_yticklabels(ylabels, fontsize=9.5)
ax2.set_xlim(0, 1.90)
ax2.set_ylim(-0.6, len(reg_order) - 0.4)
ax2.spines['left'].set_visible(False)
ax2.tick_params(left=False)
ax2.grid(axis='x', color=GRID_COL, linewidth=0.55)
ax2.grid(axis='y', visible=False)
ax2.text(1.88, -0.55,
         '* p<0.05   ** p<0.01   *** p<0.001',
         fontsize=8, color=MUTED, ha='right', style='italic')
panel_label(ax2, 'B', 'Regional β coefficients  vs.  global benchmark')
style_ax(ax2, xlabel='|β|  convergence coefficient  ±  95% CI')

# ── Row 3: bubble chart β vs R² ───────────────────────────────────────────────
ax3 = fig5.add_subplot(gs5[2])

# Threshold reference lines (drawn first, behind bubbles)
for yval, lbl, yloc in [
    (0.30, 'Moderate fit  (R² = 0.30)', 0.22),
    (0.60, 'Strong fit  (R² = 0.60)',   0.52),
]:
    ax3.axhline(yval, color=WB_LGREY, lw=0.9, linestyle=':', zorder=1)
    ax3.text(0.07, yval + 0.015, lbl, fontsize=8, color=MUTED, style='italic')

ax3.text(0.07, 0.07, 'Weak fit', fontsize=8, color=MUTED, style='italic')

# Global β reference
ax3.axvline(abs(a1), color=TEXT_COL, lw=1.2, linestyle='--',
            zorder=2, alpha=0.65)
ax3.text(abs(a1) + 0.012, 0.81,
         f'Global β = {abs(a1):.3f}',
         fontsize=8.5, color=MUTED, style='italic')

# Fixed label offsets per region to prevent any overlap
BUBBLE_OFFSETS = {
    'East Asia & Pacific':        ( 0.03,  0.03),
    'Europe & Central Asia':      ( 0.03,  0.03),
    'Latin America & Caribbean':  ( 0.03, -0.06),
    'Middle East & North Africa': ( 0.03,  0.03),
    'South Asia':                 (-0.22,  0.04),
    'Sub-Saharan Africa':         ( 0.03, -0.07),
}

for reg in reg_res:
    res  = reg_res[reg]
    col  = REGION_COLORS.get(reg, WB_GREY)
    size = max(res['N'] * 11, 90)
    bx, by = abs(res['beta']), res['r2']
    ax3.scatter(bx, by, color=col, s=size, alpha=0.85, zorder=4,
                edgecolors='white', linewidths=1.4)
    dx, dy = BUBBLE_OFFSETS.get(reg, (0.03, 0.03))
    ax3.text(bx + dx, by + dy,
             REG_SHORT[reg],
             fontsize=10, color=col, fontweight='bold', va='center')

# SSA callout
ssa = reg_res.get('Sub-Saharan Africa')
if ssa:
    ax3.annotate(
        'SSA: convergence present\nbut highly dispersed  (R² = 0.09)',
        xy=(abs(ssa['beta']), ssa['r2']),
        xytext=(0.72, 0.20),
        fontsize=8.5, color=WB_RED, style='italic',
        arrowprops=dict(arrowstyle='->', color=WB_RED, lw=1.1,
                        connectionstyle='arc3,rad=0.25'),
        bbox=dict(boxstyle='round,pad=0.35', facecolor='white',
                  edgecolor=WB_RED, linewidth=0.8, alpha=0.9),
    )

# Bubble size legend — bottom right, manually placed to avoid data
for n_ref, ypos in [(10, 0.15), (30, 0.38), (55, 0.61)]:
    ax3.scatter(1.13, ypos, color=WB_GREY, s=n_ref * 11,
                alpha=0.5, edgecolors='white', linewidths=1.0, zorder=3,
                clip_on=False)
    ax3.text(1.17, ypos, f'N = {n_ref}',
             fontsize=8.5, color=MUTED, va='center')
ax3.text(1.10, 0.70, 'Sample\nsize', fontsize=8, color=MUTED,
         style='italic', ha='center')

ax3.set_xlim(0.05, 1.25)
ax3.set_ylim(0.00, 0.86)
panel_label(ax3, 'C',
    'Convergence speed vs. model fit  '
    '— bubble size proportional to country sample')
style_ax(ax3,
    xlabel='|β|  convergence speed',
    ylabel='R²  explanatory power of initial poverty')
ax3.spines['left'].set_color(WB_LGREY)
ax3.spines['bottom'].set_color(WB_LGREY)

source_note(fig5,
    'Source: World Bank, World Development Indicators (SI.POV.DDAY). '
    'Poverty headcount ratio at $3.00/day (2021 PPP). '
    'Beta-convergence: Barro & Sala-i-Martin (1992). '
    'Annualized change in poverty regressed on log initial poverty level. '
    'Widest available observation window per country (minimum 5 years). '
    'Panel B: dashed line = global β benchmark. '
    'Panel C: dotted lines = R² fit thresholds; bubble size ∝ N countries.',
    y=0.015,
)

plt.savefig('fig5_beta_convergence.png', dpi=180, bbox_inches='tight', facecolor=WB_BG)
print("fig5 saved.")
plt.show()


# ── CELL 6 · Figure 6 — Sigma-Convergence ────────────────────────────────────
#
#   Layout: 4 rows, each full width
#     Row 1 : sigma over time + OLS trend
#     Row 2 : initial vs final scatter by income group
#     Row 3 : HP cycle amplitude + std dev by region
#     Row 4 : coefficient of variation over time
#
fig6 = plt.figure(figsize=(14, 22), facecolor=WB_BG)
gs6  = gridspec.GridSpec(
    4, 1, figure=fig6,
    left=0.09, right=0.94, top=0.95, bottom=0.05,
    hspace=0.9, # Increased hspace
    height_ratios=[1.0, 1.0, 1.0, 1.0],
)

fig6.text(0.09, 0.972,
    'Sigma-Convergence and Distributional Analysis — Poverty Rate Dispersion, 1990–2023',
    fontsize=14, fontweight='bold', color=TEXT_COL, fontfamily='DejaVu Serif')
fig6.text(0.09, 0.958,
    'Sigma-convergence measures whether cross-country dispersion of poverty rates '
    'has narrowed over time  (Sala-i-Martin, 1996).',
    fontsize=9, color=MUTED)

# Shared region legend
handles_r = [mpatches.Patch(color=REGION_COLORS[r], label=r) for r in REGION_COLORS]
fig6.legend(
    handles=handles_r,
    loc='upper right', bbox_to_anchor=(0.94, 0.968),
    ncol=2, fontsize=8.5,
    frameon=True, framealpha=0.95, edgecolor=WB_LGREY,
    handlelength=1.0,
)

# ── Row 1: sigma over time ────────────────────────────────────────────────────
ax_s1 = fig6.add_subplot(gs6[0])

yr_arr  = sigma_df['year'].values
sig_arr = sigma_df['sigma'].values

ax_s1.fill_between(yr_arr, sig_arr, alpha=0.09, color=WB_BLUE)
ax_s1.plot(yr_arr, sig_arr, color=WB_BLUE, lw=2.2, zorder=4,
           label='Cross-country σ (std. dev.)')
ax_s1.plot(yr_arr, sl_s * yr_arr + ic_s, color=WB_RED, lw=1.8,
           linestyle='--', zorder=5,
           label=f'OLS trend  slope = {sl_s:.3f} pp/yr   R² = {r_s**2:.3f}')

# Event markers
events = [(2008, 'GFC 2008'), (2015, 'MENA reversal'), (2020, 'COVID-19')]
ymax   = sig_arr.max()
for xv, lbl in events:
    ax_s1.axvline(xv, color=WB_GREY, lw=1.0, linestyle=':', alpha=0.65, zorder=2)
    ax_s1.text(xv + 0.4, ymax * 0.88, lbl,
               fontsize=8, color=MUTED, style='italic', va='top')

ax_s1.set_xlim(1988, 2025)
ax_s1.legend(fontsize=9, loc='upper right', frameon=True,
             framealpha=0.9, edgecolor=WB_LGREY)
panel_label(ax_s1, 'A',
    'Cross-country dispersion narrows over time — '
    'but disruptions widen it again after 2015')
style_ax(ax_s1,
    ylabel='Cross-country std. deviation (pp)',
    xlabel='Year')
ax_s1.spines['left'].set_color(WB_LGREY)
ax_s1.spines['bottom'].set_color(WB_LGREY)

# ── Row 2: initial vs final by income group ───────────────────────────────────
ax_s2 = fig6.add_subplot(gs6[1])

for ig, col in IG_COLORS.items():
    sub = dfc[dfc['income'] == ig]
    if len(sub) == 0: continue
    ax_s2.scatter(sub['p0'], sub['p1'],
                  color=col, alpha=0.65, s=30, label=ig,
                  zorder=4, edgecolors='none', rasterized=True)

# No-change diagonal
ax_s2.plot([0, 100], [0, 100], color=WB_LGREY, lw=1.2,
           linestyle='--', zorder=2, label='No-change line')

# Shade "below diagonal" region
ax_s2.fill_between([0, 100], [0, 100], 0,
                   color=WB_BLUE, alpha=0.03, zorder=1)

ax_s2.set_xlim(0, 105); ax_s2.set_ylim(0, 105)
ax_s2.legend(fontsize=9, loc='upper left', frameon=True,
             framealpha=0.9, edgecolor=WB_LGREY)
ax_s2.text(60, 6,
           'Below diagonal = poverty reduced',
           fontsize=8.5, color=MUTED, style='italic')
panel_label(ax_s2, 'B',
    'Initial vs. final poverty rate — most countries below the no-change diagonal')
style_ax(ax_s2,
    xlabel='Initial poverty rate  p₀ (%)',
    ylabel='Final poverty rate  p₁ (%)')
ax_s2.spines['left'].set_color(WB_LGREY)
ax_s2.spines['bottom'].set_color(WB_LGREY)

# ── Row 3: cycle amplitude + std dev ─────────────────────────────────────────
ax_s3 = fig6.add_subplot(gs6[2])

regs_s = sorted(
    reg_hp.keys(),
    key=lambda r: max(reg_hp[r]['cycle']) - min(reg_hp[r]['cycle']),
    reverse=True,
)
amps   = [max(reg_hp[r]['cycle']) - min(reg_hp[r]['cycle']) for r in regs_s]
stds   = [float(np.std(reg_hp[r]['cycle'])) for r in regs_s]
cols_s = [REGION_COLORS[r] for r in regs_s]
xlbls  = [
    r.replace('Latin America & Caribbean', 'Latin Am. & Caribbean')
     .replace('Middle East & North Africa', 'Middle East & N. Africa')
    for r in regs_s
]

x_pos3 = np.arange(len(regs_s))
width  = 0.36

bars_a = ax_s3.bar(x_pos3 - width / 2, amps,
                   width, color=cols_s, zorder=3,
                   edgecolor='none', label='Amplitude  (max − min)')
bars_s = ax_s3.bar(x_pos3 + width / 2, stds,
                   width, color=cols_s, zorder=3,
                   edgecolor='none', alpha=0.45, label='Std. deviation')

# Value labels with enough vertical clearance
ymax3 = max(amps) * 1.30
for bar, v in zip(bars_a, amps):
    ax_s3.text(bar.get_x() + bar.get_width() / 2, v + ymax3 * 0.02,
               f'{v:.1f}', ha='center', fontsize=9, color=TEXT_COL)
for bar, v in zip(bars_s, stds):
    ax_s3.text(bar.get_x() + bar.get_width() / 2, v + ymax3 * 0.02,
               f'{v:.2f}', ha='center', fontsize=8.5, color=MUTED)

ax_s3.set_xticks(x_pos3)
ax_s3.set_xticklabels(xlbls, fontsize=9)
ax_s3.set_ylim(0, ymax3)
ax_s3.legend(fontsize=9, loc='upper right', frameon=True,
             framealpha=0.9, edgecolor=WB_LGREY)
panel_label(ax_s3, 'C',
    'HP cyclical component by region — amplitude and standard deviation  (λ = 100)')
style_ax(ax_s3, ylabel='Percentage points')
ax_s3.spines['left'].set_color(WB_LGREY)
ax_s3.spines['bottom'].set_color(WB_LGREY)

# ── Row 4: coefficient of variation ──────────────────────────────────────────
ax_s4 = fig6.add_subplot(gs6[3])

ax_s4.fill_between(sigma_df['year'], sigma_df['cv'],
                   alpha=0.09, color=WB_PURPLE)
ax_s4.plot(sigma_df['year'], sigma_df['cv'],
           color=WB_PURPLE, lw=2.2, zorder=4,
           label='Coefficient of variation  (σ / μ)')

# CV = 1 reference
ax_s4.axhline(1.0, color=WB_RED, lw=1.2, linestyle='--',
              alpha=0.80, zorder=2)
ax_s4.text(1989, 1.02,
           'CV = 1.0 — std. dev. equals mean',
           fontsize=8.5, color=WB_RED, style='italic', va='bottom')

# Event markers
cv_max = sigma_df['cv'].max()
for xv, lbl in [(2015, '2015'), (2020, 'COVID-19')]:
    ax_s4.axvline(xv, color=WB_GREY, lw=1.0, linestyle=':', alpha=0.65)
    ax_s4.text(xv + 0.4, cv_max * 0.86, lbl,
               fontsize=8, color=MUTED, style='italic')

ax_s4.set_xlim(1988, 2025)
ax_s4.legend(fontsize=9, loc='upper left', frameon=True,
             framealpha=0.9, edgecolor=WB_LGREY)
panel_label(ax_s4, 'D',
    'Coefficient of variation — relative dispersion rises after 2013 '
    'despite falling absolute σ')
style_ax(ax_s4,
    ylabel='Coefficient of variation  (σ / μ)',
    xlabel='Year')
ax_s4.spines['left'].set_color(WB_LGREY)
ax_s4.spines['bottom'].set_color(WB_LGREY)

source_note(fig6,
    'Source: World Bank, World Development Indicators (SI.POV.DDAY). '
    'Poverty headcount ratio at $3.00/day (2021 PPP). '
    'Panel A: cross-country σ across countries with ≥10 observations. '
    'Panel C: HP filter (λ = 100) applied separately to each regional aggregate. '
    'Panel D: coefficient of variation = σ/μ, captures relative rather than absolute dispersion.',
    y=0.010,
)

plt.savefig('fig6_sigma_convergence.png', dpi=180, bbox_inches='tight', facecolor=WB_BG)
print("fig6 saved.")
plt.show()


# ── CELL 7 · Download figures (Colab only) ────────────────────────────────────
# Uncomment these lines when running in Google Colab:
#
# from google.colab import files
# files.download('fig5_beta_convergence.png')
# files.download('fig6_sigma_convergence.png')

## Resources

## Notebook Name

**Poverty Convergence and Forecasting Analysis**

### Fetching GDP per capita data

To analyze the relationship between poverty reduction and economic growth, we need GDP per capita data. We'll use the `wbgapi` library to fetch GDP per capita (constant 2015 US$) from the World Bank's World Development Indicators (WDI) database. The indicator code for this is `NY.GDP.PCAP.KD`.

In [ ]:
try:
    import wbgapi as wb
except ImportError:
    print("wbgapi not found. Installing...")
    !pip install wbgapi --quiet
    import wbgapi as wb

print(f"wbgapi {wb.__version__}")


In [ ]:
# Fetch GDP per capita data (constant 2015 US$)
gdp_indicator = 'NY.GDP.PCAP.KD'
gdp_df_raw = wb.data.DataFrame(gdp_indicator, mrv=30, skipAggs=True)

# Rename columns and set index
gdp_df_raw = gdp_df_raw.reset_index().rename(columns={'economy': 'Country Code'})
gdp_df_raw.columns = [col.replace('YR', '') for col in gdp_df_raw.columns]

# Pivot for easier merging: Country Code as index, years as columns
gdp_df_melted = gdp_df_raw.melt(id_vars=['Country Code'], var_name='year', value_name='gdp_per_capita')
gdp_df_melted['year'] = pd.to_numeric(gdp_df_melted['year'], errors='coerce')
gdp_df_melted = gdp_df_melted.dropna(subset=['year']).copy()

# Display the first few rows of the fetched GDP data
print("GDP per capita data head:")
display(gdp_df_melted.head())


### Merging poverty and GDP data

Now, we'll extract the initial GDP per capita for each country, corresponding to their `y_start` year in the poverty data, and then merge this with our poverty convergence DataFrame (`dfc`).

In [ ]:
# Extract initial GDP per capita for each country from dfc's y_start
def get_initial_gdp(row):
    country_code = row['code']
    start_year = row['y_start']
    gdp_value = gdp_df_melted[
        (gdp_df_melted['Country Code'] == country_code) &
        (gdp_df_melted['year'] == start_year)
    ]['gdp_per_capita'].values
    return gdp_value[0] if len(gdp_value) > 0 else np.nan

dfc['gdp_initial'] = dfc.apply(get_initial_gdp, axis=1)

# Drop rows where initial GDP per capita is missing
dfc_gdp = dfc.dropna(subset=['gdp_initial']).copy()

print(f"Merged data has {len(dfc_gdp)} countries with both poverty and initial GDP data.")
display(dfc_gdp.head())


### Scatter plot: Poverty Reduction vs. Initial GDP per Capita

Let's visualize the relationship between the annualized change in poverty and the initial GDP per capita using a scatter plot. We will color-code the points by region to identify any regional patterns.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 8))

for reg, col in REGION_COLORS.items():
    sub_df = dfc_gdp[dfc_gdp['region'] == reg]
    ax.scatter(sub_df['gdp_initial'], sub_df['ann_change'],
               color=col, alpha=0.7, s=50, label=reg, edgecolors='white', linewidths=0.5)

# Add global OLS fit line
x_gdp = dfc_gdp['gdp_initial'].values
y_change = dfc_gdp['ann_change'].values

# Apply log transform to GDP for the regression line if there's a wide range
# For visualization, we can keep original scale on x-axis but fit on log(gdp)
log_gdp_initial = np.log(dfc_gdp['gdp_initial'].values)

# Check for NaNs or infs before OLS, if any, remove them
valid_indices = np.isfinite(log_gdp_initial) & np.isfinite(y_change)
log_gdp_initial_filtered = log_gdp_initial[valid_indices]
y_change_filtered = y_change[valid_indices]

# Perform OLS on log(gdp_initial) and ann_change
if len(log_gdp_initial_filtered) > 1:
    a_gdp, b_gdp, _, _, _, p_gdp, r2_gdp, _, _ = ols(log_gdp_initial_filtered, y_change_filtered)

    # Generate points for the regression line for plotting
    # Use the original GDP range, then transform to log scale for prediction
    gdp_fit_range = np.linspace(dfc_gdp['gdp_initial'].min(), dfc_gdp['gdp_initial'].max(), 100)
    log_gdp_fit_range = np.log(gdp_fit_range)
    ann_change_fit = a_gdp + b_gdp * log_gdp_fit_range

    ax.plot(gdp_fit_range, ann_change_fit, color=TEXT_COL, linestyle='--',
            label=f'OLS Fit (log(GDP)): β = {b_gdp:.3f}, R² = {r2_gdp:.2f}, p < {p_gdp:.3f}', lw=1.5)
else:
    print("Not enough data points for OLS fit on GDP per capita.")

ax.axhline(0, color=WB_GREY, linestyle=':', lw=1)

ax.set_xscale('log') # Use log scale for x-axis for better visualization due to wide GDP range

ax.set_title('Poverty Reduction vs. Initial GDP per Capita (Log Scale)', fontsize=14, fontweight='bold')
ax.set_xlabel('Initial GDP per Capita (Constant 2015 US$, Log Scale)', fontsize=11)
ax.set_ylabel('Annualized Change in Poverty Rate (percentage points)', fontsize=11)
ax.legend(title='Region', bbox_to_anchor=(1.05, 1), loc='upper left')
ax.grid(True, linestyle='--', alpha=0.7)
plt.tight_layout()
plt.savefig('fig_poverty_vs_gdp_scatter.png', dpi=180, bbox_inches='tight', facecolor=WB_BG)
plt.show()


### Regression Analysis: Poverty Reduction vs. Initial GDP per Capita

Let's perform a formal Ordinary Least Squares (OLS) regression to analyze the relationship between the annualized change in poverty rate and the initial GDP per capita. The `ols` function defined earlier will be used for this analysis.

In [ ]:
# Perform OLS regression using the 'ols' helper function
# The x-variable will be log of initial GDP per capita, as is common for growth regressions
# The y-variable will be the annualized change in poverty rate

# Filter out NaN values from both series before regression
reg_df = dfc_gdp.dropna(subset=['gdp_initial', 'ann_change']).copy()

# Apply log transform to gdp_initial for regression
reg_df['log_gdp_initial'] = np.log(reg_df['gdp_initial'])

a_gdp_ols, b_gdp_ols, se0_gdp_ols, se1_gdp_ols, t_gdp_ols, p_gdp_ols, r2_gdp_ols, yh_gdp_ols, resid_gdp_ols = ols(
    reg_df['log_gdp_initial'].values,
    reg_df['ann_change'].values
)

print("--- Regression Results: Poverty Change vs. Log Initial GDP per Capita ---")
print(f"Intercept (a): {a_gdp_ols:.3f}")
print(f"Slope (β): {b_gdp_ols:.3f}")
print(f"R-squared: {r2_gdp_ols:.3f}")
print(f"P-value for slope: {p_gdp_ols:.3f} {sig_stars(p_gdp_ols)}")
print(f"Number of observations (N): {len(reg_df)}")

if b_gdp_ols < 0:
    print("\nInterpretation: The negative slope (β) suggests that countries with higher initial GDP per capita tend to have a faster rate of poverty reduction (or slower increase in poverty). This indicates a form of conditional convergence.")
else:
    print("\nInterpretation: The positive slope (β) suggests that countries with higher initial GDP per capita tend to have a slower rate of poverty reduction (or faster increase in poverty).")

if p_gdp_ols < 0.05:
    print("The relationship is statistically significant.")
else:
    print("The relationship is not statistically significant at the 0.05 level.")


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Create the scatter plot for poverty reduction vs. log initial GDP per capita
fig_reg_scatter, ax_reg_scatter = plt.subplots(figsize=(12, 8))

# Scatter plot for each region
for reg, col in REGION_COLORS.items():
    sub_df = reg_df[reg_df['region'] == reg]
    ax_reg_scatter.scatter(sub_df['log_gdp_initial'], sub_df['ann_change'],
                           color=col, alpha=0.7, s=50, label=reg, edgecolors='white', linewidths=0.5)

# Add the OLS regression line
ax_reg_scatter.plot(reg_df['log_gdp_initial'], yh_gdp_ols, color=WB_RED, linestyle='-',
                    label=f'OLS Fit (Log GDP): β = {b_gdp_ols:.3f}, R² = {r2_gdp_ols:.2f}, p = {p_gdp_ols:.3f}', lw=2)

# Add a horizontal line at 0 for reference
ax_reg_scatter.axhline(0, color=WB_GREY, linestyle=':', lw=1)

# Set title and labels
ax_reg_scatter.set_title('Poverty Reduction vs. Log Initial GDP per Capita (Regression)', fontsize=14, fontweight='bold')
ax_reg_scatter.set_xlabel('Log Initial GDP per Capita', fontsize=11)
ax_reg_scatter.set_ylabel('Annualized Change in Poverty Rate (percentage points)', fontsize=11)

# Add legend
ax_reg_scatter.legend(title='Region', bbox_to_anchor=(1.05, 1), loc='upper left')

# Add grid for better readability
ax_reg_scatter.grid(True, linestyle='--', alpha=0.7)

plt.tight_layout()
plt.savefig('fig_poverty_vs_log_gdp_regression_scatter.png', dpi=180, bbox_inches='tight', facecolor=WB_BG)
plt.show()

In [ ]:
# @title step_artifacts
num_fig = "8" # Assuming this is the 8th figure
step = 'RegressionAnalysis' # Or 'CorrelationAnalysis'
# If `upload_plt_to_gcs` is defined elsewhere, uncomment the line below:
# upload_plt_to_gcs(num_fig, step, fig_reg_scatter)
print("fig_poverty_vs_log_gdp_regression_scatter.png saved.")

### ARIMA Forecasting Setup

ARIMA models are a class of models used for forecasting time series data. They require the time series to be stationary, meaning its statistical properties (like mean and variance) do not change over time. If the data is not stationary, differencing (the 'I' in ARIMA) can be applied.

In [ ]:
# Import necessary libraries for ARIMA
import statsmodels.api as sm
from statsmodels.tsa.stattools import adfuller

print(f"statsmodels {sm.__version__}")

### 1. Data Preparation: Inspecting the `sigma` time series

We will use the `sigma` column from the `sigma_df` for ARIMA forecasting. Let's first inspect its structure and visualize it.

In [ ]:
print("Sigma DataFrame Head:")
display(sigma_df.head())

print("Sigma DataFrame Info:")
sigma_df.info()

In [ ]:
import matplotlib.pyplot as plt

# Plot the 'sigma' time series
plt.figure(figsize=(12, 6))
plt.plot(sigma_df['year'], sigma_df['sigma'], marker='o', linestyle='-', color=WB_BLUE)
plt.title('Cross-Country Standard Deviation of Poverty Rates (Sigma) Over Time', fontsize=14, fontweight='bold')
plt.xlabel('Year', fontsize=11)
plt.ylabel('Sigma (Standard Deviation)', fontsize=11)
plt.grid(True, linestyle='--', alpha=0.7)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

### 2. Check for Stationarity

ARIMA models assume that the time series is stationary. We can use the Augmented Dickey-Fuller (ADF) test to check for stationarity. If the p-value is high (e.g., > 0.05), it suggests the series is non-stationary and may require differencing.

In [ ]:
from statsmodels.tsa.stattools import adfuller

def adf_test(series):
    result = adfuller(series.dropna())
    print('ADF Statistic: %f' % result[0])
    print('p-value: %f' % result[1])
    print('Critical Values:')
    for key, value in result[4].items():
        print('\t%s: %.3f' % (key, value))
    if result[1] <= 0.05:
        print("Conclusion: The series is likely stationary.")
    else:
        print("Conclusion: The series is likely non-stationary and may need differencing.")

print("ADF test for original 'sigma' series:")
adf_test(sigma_df['sigma'])

### 3. Differencing the series and re-checking Stationarity

Since the original series is non-stationary, we'll apply a first-order differencing (d=1) and then re-run the ADF test.

In [ ]:
# Apply first-order differencing
sigma_df['sigma_diff'] = sigma_df['sigma'].diff().dropna()

print("ADF test for differenced 'sigma' series:")
adf_test(sigma_df['sigma_diff'])

### 4. Plot ACF and PACF to determine p and q

Once the series is stationary, we can use ACF (Autocorrelation Function) and PACF (Partial Autocorrelation Function) plots to determine the `p` (AR order) and `q` (MA order) for the ARIMA model.

*   **ACF plot**: Helps determine the moving average (MA) order `q`. Look for the point where the ACF plot drops off sharply to zero.
*   **PACF plot**: Helps determine the autoregressive (AR) order `p`. Look for the point where the PACF plot drops off sharply to zero.

In [ ]:
import pandas as pd
import statsmodels.api as sm
import itertools
import warnings

# Set 'year' as index and convert it to DatetimeIndex for time series modeling
sigma_ts = sigma_df.set_index('year')['sigma']
sigma_ts.index = pd.to_datetime(sigma_ts.index, format='%Y')

# Define the p, d, q parameters to take any value between 0 and 2
p = d = q = range(0, 3)

# Generate all different combinations of p, d, and q triplets
pdq_combinations = list(itertools.product(p, d, q))

best_aic = float("inf")
best_order = None
best_results = None

print('Searching for optimal ARIMA model parameters (p, d, q)...')

# Iterate over all parameter combinations and fit ARIMA model
for order in pdq_combinations:
    try:
        # Suppress convergence warnings
        with warnings.catch_warnings():
            warnings.filterwarnings("ignore")
            model = sm.tsa.arima.ARIMA(sigma_ts, order=order)
            results = model.fit()

        if results.aic < best_aic:
            best_aic = results.aic
            best_order = order
            best_results = results
    except Exception as e:
        # print(f"Error fitting ARIMA {order}: {e}") # Uncomment for debugging
        continue

print(f'Optimal ARIMA(p,d,q) order: {best_order} with AIC: {best_aic:.2f}')
print('\nBest ARIMA Model Summary:')
print(best_results.summary())

In [ ]:
import matplotlib.pyplot as plt

# Get forecast out-of-sample using the best_results
# Forecast for the next 5 years after the last observed year.
last_year = sigma_ts.index.max()
forecast_start = last_year + pd.DateOffset(years=1)
forecast_end = last_year + pd.DateOffset(years=5)

# Generate forecast with confidence intervals
# The get_forecast method of SARIMAXResults object automatically handles DatetimeIndex
forecast_obj = best_results.get_forecast(steps=5)
forecast = forecast_obj.predicted_mean
conf_int = forecast_obj.conf_int()

# Plot the results
plt.figure(figsize=(14, 7))
plt.plot(sigma_ts.index, sigma_ts.values, label='Original Sigma Data', color=WB_BLUE, marker='o', linestyle='-')
plt.plot(best_results.fittedvalues.index, best_results.fittedvalues.values, label=f'ARIMA{best_order} Fitted Values', color=WB_RED, linestyle='--')
plt.plot(forecast.index, forecast.values, label=f'ARIMA{best_order} Forecast', color=WB_GREEN, marker='^', linestyle='-')
plt.fill_between(conf_int.index, conf_int.iloc[:, 0], conf_int.iloc[:, 1], color=WB_GREEN, alpha=0.2, label='95% Confidence Interval')

plt.title(f'ARIMA{best_order} Forecasting of Cross-Country Sigma (Standard Deviation of Poverty Rates)', fontsize=14, fontweight='bold')
plt.xlabel('Year', fontsize=11)
plt.ylabel('Sigma (Standard Deviation)', fontsize=11)
plt.legend()
plt.grid(True, linestyle='--', alpha=0.7)
plt.xticks(rotation=45)
plt.tight_layout()

plt.savefig('fig_arima_sigma_forecast.png', dpi=180, bbox_inches='tight', facecolor=WB_BG)
plt.show()

In [ ]:
# @title step_artifacts
num_fig = '9' # Assuming this is the 9th figure
step = 'ARIMAForecasting'
# If `upload_plt_to_gcs` is defined elsewhere, uncomment the line below:
# upload_plt_to_gcs(num_fig, step, fig_arima_sigma_forecast)
print("fig_arima_sigma_forecast.png saved.")

## Policy Analysis: Implications for Poverty Reduction Strategies

The analysis of poverty convergence, its relationship with GDP per capita, and future trends of dispersion provides critical insights for policymakers aiming to enhance poverty reduction strategies.

### Key Findings and Policy Implications:

1.  **Strong Beta-Convergence**: The statistically significant negative beta coefficient (β = -0.409, p < 0.001) confirms **beta-convergence**, meaning countries with higher initial poverty levels tend to experience faster rates of poverty reduction. This is a positive indicator that targeted interventions in the poorest countries can yield substantial results. However, the varying regional beta coefficients suggest that "one-size-fits-all" policies may be ineffective. Policies should be tailored to regional contexts, acknowledging differences in convergence speeds and underlying drivers.

2.  **Sigma-Convergence with Disruptions**: While cross-country standard deviation (sigma) of poverty rates showed a general narrowing trend, indicating **sigma-convergence**, this trend was disrupted after 2015, leading to a widening of dispersion. The rise in the coefficient of variation after 2013 further highlights an increase in relative inequality. This implies that while some countries are catching up, others are falling behind, exacerbating global poverty disparities. Policymakers must focus on policies that not only reduce poverty in individual nations but also mitigate the growing divergence between countries, potentially through enhanced global aid, technology transfer, and fairer trade policies.

3.  **Poverty Reduction vs. Initial GDP per Capita**: A statistically significant positive slope (β = 0.389, p = 0.013) indicates that countries with higher initial GDP per capita tended to experience a *slower* rate of poverty reduction. This counter-intuitive finding challenges the assumption that higher initial wealth automatically translates to faster poverty eradication. It suggests that economic growth alone is insufficient; policies must ensure **inclusive growth** that directly benefits the poor. This could involve progressive taxation, social safety nets, investments in human capital (education, health), and equitable access to resources and opportunities, particularly in countries transitioning to higher income brackets.

4.  **ARIMA Forecast for Sigma Convergence (ARIMA(0, 2, 2))**: The optimal ARIMA(0, 2, 2) model forecasts a gradual increase in the cross-country standard deviation of poverty rates over the next five years, with widening confidence intervals. This forecast underscores the projected increase in global poverty divergence. This dire outlook necessitates proactive measures. Policymakers should:
    *   **Prioritize Fragile States and Vulnerable Populations**: Direct resources and support to countries and communities most at risk of falling further behind.
    *   **Strengthen Resilience**: Implement policies that build economic and social resilience against shocks (e.g., climate change, pandemics, economic crises) that disproportionately affect poorer nations and can reverse progress in poverty reduction.
    *   **Foster International Cooperation**: Enhance international partnerships and multilateral efforts to address global challenges that contribute to poverty disparities.

### Overall Recommendation:

Future poverty reduction strategies should move beyond mere aggregate reduction targets and embrace a **two-pronged approach**: actively fostering convergence in lagging regions through targeted support, while simultaneously ensuring that economic growth in wealthier developing nations is inclusive and translates into genuine poverty eradication. Monitoring sigma-convergence trends and adapting policies to address emerging disparities will be crucial for achieving sustainable and equitable global poverty reduction.

In [40]:
!jupyter nbconvert --ClearOutputPreprocessor.enabled=True --inplace Initial_commitment_poverty_reduction.ipynb

[NbConvertApp] WARNING | pattern 'Initial_commitment_poverty_reduction.ipynb' matched no files
This application is used to convert notebook files (*.ipynb)
        to various other formats.


Options
The options below are convenience aliases to configurable class-options,
as listed in the "Equivalent to" description-line of the aliases.
To see all configurable class-options for some <cmd>, use:
    <cmd> --help-all

--debug
    set log level to logging.DEBUG (maximize logging output)
    Equivalent to: [--Application.log_level=10]
--show-config
    Show the application's configuration (human-readable format)
    Equivalent to: [--Application.show_config=True]
--show-config-json
    Show the application's configuration (json format)
    Equivalent to: [--Application.show_config_json=True]
--generate-config
    generate default config file
    Equivalent to: [--JupyterApp.generate_config=True]
-y
    Answer yes to any questions instead of prompting.
    Equivalent to: [--JupyterApp.answe